In [1]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import os


In [2]:
song_dataset_path = os.path.join("data","Music Info.csv")

In [3]:
df_song = pd.read_csv(song_dataset_path)

In [4]:
# Dropping the duplicated

df_song.drop_duplicates(subset=['spotify_id','year','duration_ms'],inplace=True)

In [5]:
df_song.reset_index(drop=True,inplace=True)

In [6]:
# remove columns not required for content based filtering 

col_to_remove = ['track_id','name','spotify_preview_url','spotify_id','genre']
df_content_filtering = df_song.drop(columns=col_to_remove)
df_content_filtering.head(2)

    


,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.0,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.0,0.2070,0.651,174.426,4


In [7]:
from sklearn.preprocessing import MinMaxScaler,StandardScaler,OneHotEncoder
from category_encoders import CountEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np
import pandas as pd

# Custom wrapper for TfidfVectorizer to work with ColumnTransformer
class TfidfVectorizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, max_features=85):
        self.max_features = max_features
        self.vectorizer = TfidfVectorizer(max_features=max_features)
    
    def fit(self, X, y=None):
        # Convert to 1D array if needed
        if isinstance(X, np.ndarray):
            if X.ndim == 2:
                X = X.ravel()
        elif hasattr(X, 'values'):  # pandas DataFrame/Series
            X = X.values.ravel()
        
        # Replace NaN/None values with empty strings
        X = pd.Series(X).fillna('').values
        
        self.vectorizer.fit(X)
        return self
    
    def transform(self, X):
        # Convert to 1D array if needed
        if isinstance(X, np.ndarray):
            if X.ndim == 2:
                X = X.ravel()
        elif hasattr(X, 'values'):  # pandas DataFrame/Series
            X = X.values.ravel()
        
        # Replace NaN/None values with empty strings
        X = pd.Series(X).fillna('').values
        
        return self.vectorizer.transform(X).toarray()

In [8]:
df_content_filtering.columns

Index(['artist', 'tags', 'year', 'duration_ms', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature'],
      dtype='str')

In [9]:
# Cols to transform

frequency_encode_cols = ['year']
ohe_cols = ['artist','time_signature','key']
tfidf_col = ['tags']
standard_scale_cols = ['duration_ms','loudness','tempo']
min_max_scale_cols = ['danceability','energy','speechiness','acousticness','instrumentalness','liveness','valence']

In [10]:
len(frequency_encode_cols+ohe_cols+standard_scale_cols+min_max_scale_cols)

14

In [11]:
# Transform the data

transformer = ColumnTransformer(transformers=[
    ("frequency_encode",CountEncoder(normalize=True,return_df=True),frequency_encode_cols),
    ("ohe",OneHotEncoder(handle_unknown="ignore"),ohe_cols),
    ("tfidf",TfidfVectorizerTransformer(max_features=85),tfidf_col),
    ("standard_scale",StandardScaler(),standard_scale_cols),
    ("min_max_scale",MinMaxScaler(),min_max_scale_cols)],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False)
transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [12]:
transformer.fit(df_content_filtering)

c:\Users\adity\Desktop\Recommandation System 2\venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [13]:
transformed_df = transformer.transform(df_content_filtering)

In [14]:
transformed_df.shape

(50674, 8431)

In [15]:
transformed_df

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 907791 stored elements and shape (50674, 8431)>

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
#fetch a song where artist is Shakira

df_song.loc[df_song['artist'] == 'Shakira'][:1]

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [18]:
df_song[df_song['name'] == 'Whenever, Wherever']

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [19]:
song_input = df_content_filtering[df_song['name'] == 'Whenever, Wherever']
song_input

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,Shakira,"rock, pop, female_vocalists, singer_songwriter...",2012,196826,0.787,0.828,1,-4.967,0,0.0474,0.298,0.000005,0.206,0.86,107.674,4


In [20]:
# Input vector to calcuate similarity
input_vector = transformer.transform(song_input)
input_vector

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20 stored elements and shape (1, 8431)>

In [21]:
similarity_score = cosine_similarity(transformed_df,input_vector)
similarity_score

array([[0.99999914],
       [0.99999847],
       [0.99999921],
       ...,
       [0.99999877],
       [0.99999932],
       [0.99999891]], shape=(50674, 1))

In [22]:
top_10_songs_indexes = np.argsort(similarity_score.ravel())[-11:][::-1]

In [23]:
top_10_songs_indexes

array([ 1025, 12305,  6046,  6129, 17241,  6133,  7172,  6121,  6526,
       38383,  6287])

In [24]:
top_10_song_names = df_song.iloc[top_10_songs_indexes]
top_10_song_names

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.29800,0.000005,0.2060,0.860,107.674,4
12305,TRYFVKK128F4240FE8,Why Wait,Shakira,https://p.scdn.co/mp3-preview/d78c90c5cb5626be...,0HiJFRxWme9myvUiDlqQ8q,"pop, experimental, singer_songwriter, dance",NaN,2001,221240,0.887,...,1,-5.535,0,0.0431,0.14400,0.000590,0.1230,0.399,129.943,4
6046,TRAAKDG128F42A0ECB,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,01Yj2MCGpjZs34PRlGgz4K,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,217453,0.777,...,10,-5.867,0,0.0734,0.28400,0.000000,0.4300,0.760,100.003,4
6129,TRBAUVN128F932FEF8,Oops!...I Did It Again,Britney Spears,https://p.scdn.co/mp3-preview/7fb86827422540ad...,095uakqDYR50Uza0mxvPWB,"pop, female_vocalists, dance, 00s",Pop,2014,211786,0.751,...,1,-5.351,0,0.0435,0.34000,0.000018,0.2550,0.886,95.045,4
17241,TRMBDIR128F4279C1F,Perfect Lover,Britney Spears,https://p.scdn.co/mp3-preview/52671e54d36f077e...,1BhxPx4evrx8X02RHGrLdi,"pop, dance, rnb, 00s",Rock,2007,182680,0.718,...,1,-3.959,0,0.0360,0.35300,0.000390,0.1020,0.805,117.067,4
6133,TRDXCSH128F92ED4A1,Bootylicious,Destiny's Child,https://p.scdn.co/mp3-preview/7e327ccb1e4c52b2...,09mkdGhqb5ySKVsnkx9hy2,"pop, female_vocalists, dance, soul, hip_hop, r...",NaN,2001,207733,0.835,...,1,-4.364,0,0.2840,0.00247,0.000000,0.1710,0.586,103.358,4
7172,TRLQBEN128F92E7415,Wild Things,Alessia Cara,https://p.scdn.co/mp3-preview/c13f00088525d0b2...,0RgHkNpqtAHGGBVro6mmqD,"pop, female_vocalists",Country,2016,188493,0.741,...,1,-5.362,0,0.1080,0.01950,0.000000,0.0822,0.735,107.960,4
6121,TRGZIMZ128F930A016,La Isla Bonita,Madonna,https://p.scdn.co/mp3-preview/d8f3cafe99c1f0cd...,0rpndqrkU9y9nckNCfjcq6,"pop, female_vocalists, dance, 80s",NaN,2009,242946,0.708,...,1,-4.736,0,0.0362,0.39200,0.000001,0.0561,0.968,99.953,4
6526,TRXHWIA128E078A9BB,Cruel Summer,Bananarama,https://p.scdn.co/mp3-preview/47d13ef240a58bef...,0BmUCeyXpTrUVahHKVFxuB,"pop, female_vocalists, dance, 80s, new_wave",NaN,1984,209573,0.665,...,1,-5.828,0,0.0278,0.25000,0.008550,0.0537,0.932,108.429,4
38383,TRWUWRZ128F42ADA4A,Dreams for Plans,Shakira,https://p.scdn.co/mp3-preview/6e2c021846087a88...,2ObxMmMaDINr0ynkqW2BlY,"pop, female_vocalists, guitar, pop_rock",Pop,2005,242760,0.689,...,1,-7.427,1,0.0286,0.18000,0.000038,0.0844,0.548,96.098,4


In [25]:
def recommend_song(song_name, songs_data, transformed_data, k):
    """
    Recommend top k songs similar to the given song using content-based filtering.
    
    Parameters:
    -----------
    song_name : str
        Name of the song to find recommendations for
    songs_data : pd.DataFrame
        Original dataframe containing song information (must have 'name', 'artist', 'spotify_id' columns)
    transformed_data : array-like
        Transformed feature matrix (output from transformer.transform())
    k : int
        Number of recommendations to return
    
    Returns:
    --------
    pd.DataFrame
        Dataframe containing top k recommended songs with columns: name, artist, spotify_url
    """
    
    # Filter songs_data for the input song
    song_mask = songs_data['name'] == song_name
    
    if not song_mask.any():
        print(f"Song '{song_name}' not found in the dataset.")
        return pd.DataFrame()
    
    # Get the index of the input song
    song_index = songs_data[song_mask].index[0]
    
    # Get transformed vector for the input song
    input_vector = transformed_data[song_index].reshape(1, -1)
    
    # Calculate cosine similarity with all songs
    similarity_scores = cosine_similarity(transformed_data, input_vector).ravel()
    
    # Get indices of top k similar songs (O(n log n))
    top_indices = np.argsort(similarity_scores)[::-1][:k]
    
    # Get the recommended songs
    recommended_songs = songs_data.iloc[top_indices].copy()
    
    # Return only name, artist, and spotify_url columns
    return recommended_songs[['name', 'artist', 'spotify_preview_url','tags']].reset_index(drop=True)


In [26]:
recommend_song("Hips Don't Lie",df_song,transformed_df,k=10)

,name,artist,spotify_preview_url,tags
0,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,"pop, female_vocalists, singer_songwriter, danc..."
1,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,"rock, pop, female_vocalists, singer_songwriter..."
2,My Prerogative,Britney Spears,https://p.scdn.co/mp3-preview/9140e378563ce3e0...,"pop, female_vocalists, dance, cover"
3,Aloha,Møme,https://p.scdn.co/mp3-preview/c4c1768c0f2beab9...,"pop, dance"
4,Say My Name,David Guetta,https://p.scdn.co/mp3-preview/89e0ed2f21273e77...,"electronic, pop, dance"
5,Always Too Late,Annie,https://p.scdn.co/mp3-preview/7f2f89b82ed206ea...,"electronic, pop, female_vocalists, dance"
6,Hoedown Throwdown,Miley Cyrus,https://p.scdn.co/mp3-preview/09a6a3b11b6f628d...,"pop, dance, soundtrack, country, 00s"
7,I'm Outta Love,Anastacia,https://p.scdn.co/mp3-preview/feda5c101a29c254...,"pop, female_vocalists, dance"
8,Whatever U Like,Nicole Scherzinger,https://p.scdn.co/mp3-preview/6b3e93d2c5472ea7...,"pop, female_vocalists, dance, hip_hop, america..."
9,AM to PM,Christina Milian,https://p.scdn.co/mp3-preview/50d697c3c682962d...,"pop, female_vocalists, dance, hip_hop, rnb"
